In [ ]:
"""
Student Collaboration Platform
A modern, dark-themed Tkinter desktop application with authentication,
WhatsApp-style chat, events, and cultural learning features.
"""

import tkinter as tk
from tkinter import ttk, scrolledtext
import threading
import time
from datetime import datetime
import random

# ─────────────────────────────────────────────
# GLOBAL THEME & STYLE CONSTANTS
# ─────────────────────────────────────────────
COLORS = {
    "bg_dark":      "#0F1117",
    "bg_card":      "#1A1D2E",
    "bg_input":     "#252840",
    "bg_hover":     "#2E3155",
    "accent":       "#6C63FF",
    "accent_light": "#8B85FF",
    "accent_dim":   "#3D3880",
    "success":      "#25D366",
    "success_dark": "#128C7E",
    "danger":       "#FF5C5C",
    "text_primary": "#EAEAEA",
    "text_muted":   "#7C7F9E",
    "text_dim":     "#4A4D6A",
    "bubble_self":  "#25D366",
    "bubble_other": "#1E2035",
    "bubble_self_text":  "#0D1F0D",
    "bubble_other_text": "#EAEAEA",
    "sidebar":      "#13152A",
    "border":       "#2A2D4A",
}

FONTS = {
    "title":   ("Segoe UI", 22, "bold"),
    "heading": ("Segoe UI", 14, "bold"),
    "body":    ("Segoe UI", 10),
    "body_b":  ("Segoe UI", 10, "bold"),
    "small":   ("Segoe UI", 8),
    "mono":    ("Consolas", 10),
    "avatar":  ("Segoe UI", 11, "bold"),
    "bubble":  ("Segoe UI", 10),
    "logo":    ("Segoe UI", 18, "bold"),
}


# ─────────────────────────────────────────────
# DATA STORE  (in-memory)
# ─────────────────────────────────────────────
class DataStore:
    """Central in-memory data repository."""

    users: dict = {}          # username → {password, display_name}
    messages: list = []       # list of message dicts
    current_user: str = None

    EVENTS = [
        {"title": "Global AI Summit 2025",        "date": "May 12", "loc": "Berlin, Germany",       "emoji": "🤖", "desc": "Leaders in AI gather to shape the future of intelligent systems."},
        {"title": "World Culture Fest",            "date": "Jun 3",  "loc": "Lagos, Nigeria",        "emoji": "🌍", "desc": "A vibrant celebration of music, food, and tradition from 50 nations."},
        {"title": "International Hackathon",       "date": "Jun 20", "loc": "San Francisco, USA",    "emoji": "💻", "desc": "48-hour innovation marathon with $100K in prizes."},
        {"title": "Climate Youth Conference",      "date": "Jul 8",  "loc": "Stockholm, Sweden",     "emoji": "🌿", "desc": "Young activists unite to accelerate climate action globally."},
        {"title": "Pan-African Student Forum",     "date": "Aug 1",  "loc": "Nairobi, Kenya",        "emoji": "🎓", "desc": "Empowering the next generation of African thought leaders."},
        {"title": "STEM Olympiad 2025",            "date": "Sep 15", "loc": "Tokyo, Japan",          "emoji": "🔬", "desc": "Elite students compete in science, technology, and mathematics."},
    ]

    CULTURES = [
        {"country": "Japan",        "flag": "🇯🇵", "greeting": "Konnichiwa!",    "fact": "Japan has over 6,800 islands and is home to the world's oldest company, founded in 578 AD.", "color": "#FF4757"},
        {"country": "Nigeria",      "flag": "🇳🇬", "greeting": "Sannu!",         "fact": "Nigeria is Africa's most populous nation with over 500 languages spoken — the most in any country.", "color": "#2ED573"},
        {"country": "Brazil",       "flag": "🇧🇷", "greeting": "Olá!",           "fact": "Brazil contains 60% of the Amazon Rainforest and has won the FIFA World Cup 5 times.", "color": "#FFA502"},
        {"country": "India",        "flag": "🇮🇳", "greeting": "Namaste!",       "fact": "India invented the number zero, chess, and yoga — shaping civilization for millennia.", "color": "#FF6B81"},
        {"country": "Germany",      "flag": "🇩🇪", "greeting": "Hallo!",         "fact": "Germany has over 1,500 different types of beer and 300+ varieties of bread.", "color": "#747D8C"},
        {"country": "South Korea",  "flag": "🇰🇷", "greeting": "Annyeong!",      "fact": "South Korea has the world's fastest internet and is ranked #1 in digital competitiveness.", "color": "#5352ED"},
        {"country": "Egypt",        "flag": "🇪🇬", "greeting": "Ahlan!",         "fact": "Ancient Egypt developed one of humanity's first writing systems — hieroglyphics — over 5,000 years ago.", "color": "#ECCC68"},
        {"country": "Mexico",       "flag": "🇲🇽", "greeting": "¡Hola!",         "fact": "Mexico City is built on an ancient lake and sinks about 10 inches every year.", "color": "#FF6348"},
    ]

    @classmethod
    def add_message(cls, sender: str, text: str):
        cls.messages.append({
            "sender":    sender,
            "text":      text,
            "timestamp": datetime.now().strftime("%I:%M %p"),
            "delivered": True,
        })

    @classmethod
    def seed_demo_data(cls):
        """Pre-load a couple of demo users and starter messages."""
        cls.users["alice"] = {"password": "alice123", "display_name": "Alice Chen"}
        cls.users["bob"]   = {"password": "bob123",   "display_name": "Bob Okafor"}
        cls.add_message("alice", "Hey everyone! 👋 Welcome to the Student Collab Platform!")
        cls.add_message("bob",   "This is so cool! Finally a place for all of us to connect 🚀")
        cls.add_message("alice", "Don't forget to check the Worldwide Events tab — there's an awesome hackathon coming up!")


# ─────────────────────────────────────────────
# HELPER UTILITIES
# ─────────────────────────────────────────────
def get_initials(name: str) -> str:
    parts = name.strip().split()
    if len(parts) >= 2:
        return (parts[0][0] + parts[-1][0]).upper()
    return name[:2].upper() if name else "??"


def make_avatar_canvas(parent, name: str, size=36, bg=None) -> tk.Canvas:
    """Draw a circular avatar with user initials."""
    AVATAR_COLORS = ["#6C63FF", "#FF6384", "#36A2EB", "#FF9F40", "#4BC0C0", "#9966FF", "#FF6348", "#2ED573"]
    color = AVATAR_COLORS[hash(name) % len(AVATAR_COLORS)]
    c = tk.Canvas(parent, width=size, height=size,
                  bg=bg or COLORS["bg_card"],
                  highlightthickness=0, bd=0)
    c.create_oval(1, 1, size - 1, size - 1, fill=color, outline="")
    c.create_text(size // 2, size // 2, text=get_initials(name),
                  fill="white", font=("Segoe UI", size // 3 + 1, "bold"))
    return c


def styled_button(parent, text, command, style="accent", width=None):
    """Return a ttk.Button with hover-effect binding."""
    btn = ttk.Button(parent, text=text, command=command,
                     style=f"{style}.TButton",
                     width=width)
    return btn


# ─────────────────────────────────────────────
# MAIN APPLICATION CONTROLLER
# ─────────────────────────────────────────────
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("StudyLink — Student Collaboration Platform")
        self.geometry("1100x720")
        self.minsize(900, 620)
        self.configure(bg=COLORS["bg_dark"])
        self._setup_styles()
        DataStore.seed_demo_data()

        # Container that holds all pages
        self.container = tk.Frame(self, bg=COLORS["bg_dark"])
        self.container.pack(fill="both", expand=True)
        self.container.grid_rowconfigure(0, weight=1)
        self.container.grid_columnconfigure(0, weight=1)

        self.pages: dict = {}
        for PageClass in (LoginPage, RegisterPage, MainApp):
            page = PageClass(self.container, self)
            self.pages[PageClass.__name__] = page
            page.grid(row=0, column=0, sticky="nsew")

        self.show_page("LoginPage")

    def show_page(self, name: str):
        page = self.pages[name]
        page.tkraise()
        if hasattr(page, "on_show"):
            page.on_show()

    def _setup_styles(self):
        style = ttk.Style(self)
        style.theme_use("clam")

        style.configure(".",
                         background=COLORS["bg_dark"],
                         foreground=COLORS["text_primary"],
                         fieldbackground=COLORS["bg_input"],
                         troughcolor=COLORS["bg_card"],
                         bordercolor=COLORS["border"],
                         darkcolor=COLORS["bg_dark"],
                         lightcolor=COLORS["bg_card"],
                         selectbackground=COLORS["accent"],
                         selectforeground="white",
                         font=FONTS["body"])

        # ── Entry ──
        style.configure("TEntry",
                         fieldbackground=COLORS["bg_input"],
                         foreground=COLORS["text_primary"],
                         insertcolor=COLORS["text_primary"],
                         borderwidth=0, relief="flat",
                         padding=(12, 8))
        style.map("TEntry", fieldbackground=[("focus", COLORS["bg_hover"])])

        # ── Label ──
        style.configure("TLabel",
                         background=COLORS["bg_dark"],
                         foreground=COLORS["text_primary"])
        style.configure("Card.TLabel",   background=COLORS["bg_card"])
        style.configure("Muted.TLabel",  background=COLORS["bg_dark"],
                         foreground=COLORS["text_muted"], font=FONTS["small"])
        style.configure("Heading.TLabel",
                         background=COLORS["bg_card"],
                         foreground=COLORS["text_primary"],
                         font=FONTS["heading"])
        style.configure("Sidebar.TLabel",
                         background=COLORS["sidebar"],
                         foreground=COLORS["text_primary"])
        style.configure("SidebarMuted.TLabel",
                         background=COLORS["sidebar"],
                         foreground=COLORS["text_muted"],
                         font=FONTS["small"])

        # ── Buttons ──
        for name, bg, fg, hov in [
            ("accent",  COLORS["accent"],       "white",               COLORS["accent_light"]),
            ("success", COLORS["success"],      COLORS["bubble_self_text"], COLORS["success_dark"]),
            ("ghost",   COLORS["bg_card"],      COLORS["text_primary"], COLORS["bg_hover"]),
            ("danger",  COLORS["danger"],       "white",               "#FF8080"),
            ("sidebar", COLORS["sidebar"],      COLORS["text_muted"],  COLORS["bg_hover"]),
            ("sidebarActive", COLORS["accent_dim"], COLORS["text_primary"], COLORS["accent"]),
        ]:
            style.configure(f"{name}.TButton",
                             background=bg, foreground=fg,
                             borderwidth=0, relief="flat",
                             padding=(14, 9), font=FONTS["body_b"],
                             focusthickness=0)
            style.map(f"{name}.TButton",
                      background=[("active", hov), ("pressed", hov)],
                      foreground=[("active", "white")])

        # ── Frame ──
        style.configure("Card.TFrame",  background=COLORS["bg_card"],  relief="flat")
        style.configure("Dark.TFrame",  background=COLORS["bg_dark"],  relief="flat")
        style.configure("Sidebar.TFrame", background=COLORS["sidebar"], relief="flat")

        # ── Scrollbar ──
        style.configure("TScrollbar",
                         background=COLORS["bg_card"],
                         troughcolor=COLORS["bg_dark"],
                         arrowcolor=COLORS["text_muted"],
                         borderwidth=0, width=6)
        style.map("TScrollbar", background=[("active", COLORS["accent_dim"])])


# ─────────────────────────────────────────────
# PAGE BASE CLASS
# ─────────────────────────────────────────────
class BasePage(ttk.Frame):
    def __init__(self, parent, controller: App):
        super().__init__(parent, style="Dark.TFrame")
        self.controller = controller

    def clear_form(self):
        """Override in subclass to reset form fields."""
        pass


# ─────────────────────────────────────────────
# LOGIN PAGE
# ─────────────────────────────────────────────
class LoginPage(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._build()

    def _build(self):
        # Centered card
        self.grid_columnconfigure(0, weight=1)
        self.grid_rowconfigure(0, weight=1)

        card = ttk.Frame(self, style="Card.TFrame", padding=50)
        card.grid(row=0, column=0)
        card.configure(width=440)

        # Logo / brand
        logo_f = ttk.Frame(card, style="Card.TFrame")
        logo_f.pack(pady=(0, 30))
        ttk.Label(logo_f, text="📚 StudyLink",
                  font=FONTS["title"],
                  foreground=COLORS["accent"],
                  background=COLORS["bg_card"]).pack()
        ttk.Label(logo_f, text="Connecting students worldwide",
                  font=FONTS["small"],
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"]).pack()

        # Fields
        self._field(card, "Username", "username")
        self._field(card, "Password", "password", show="●")

        # Error label
        self.error_var = tk.StringVar()
        ttk.Label(card, textvariable=self.error_var,
                  foreground=COLORS["danger"],
                  background=COLORS["bg_card"],
                  font=FONTS["small"]).pack(pady=(4, 0))

        # Login button
        styled_button(card, "Login →", self._login, style="accent", width=30).pack(pady=(18, 6))

        # Register link
        link_f = ttk.Frame(card, style="Card.TFrame")
        link_f.pack()
        ttk.Label(link_f, text="New here?",
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["body"]).pack(side="left")
        lnk = tk.Label(link_f, text=" Create an account",
                        fg=COLORS["accent_light"],
                        bg=COLORS["bg_card"],
                        font=("Segoe UI", 10, "underline"),
                        cursor="hand2")
        lnk.pack(side="left")
        lnk.bind("<Button-1>", lambda e: self.controller.show_page("RegisterPage"))

    def _field(self, parent, label: str, attr: str, show=""):
        ttk.Label(parent, text=label,
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["small"]).pack(anchor="w", pady=(10, 2))
        var = tk.StringVar()
        entry = ttk.Entry(parent, textvariable=var, show=show, width=32)
        entry.pack(ipady=6, fill="x")
        setattr(self, f"{attr}_var", var)

    def _login(self):
        uname = self.username_var.get().strip().lower()
        pwd   = self.password_var.get().strip()
        user  = DataStore.users.get(uname)
        if not uname or not pwd:
            self.error_var.set("Please fill in all fields.")
            return
        if not user or user["password"] != pwd:
            self.error_var.set("Invalid username or password.")
            return
        DataStore.current_user = uname
        self.error_var.set("")
        self.controller.show_page("MainApp")

    def on_show(self):
        self.error_var.set("")
        self.username_var.set("")
        self.password_var.set("")


# ─────────────────────────────────────────────
# REGISTER PAGE
# ─────────────────────────────────────────────
class RegisterPage(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._build()

    def _build(self):
        self.grid_columnconfigure(0, weight=1)
        self.grid_rowconfigure(0, weight=1)

        card = ttk.Frame(self, style="Card.TFrame", padding=50)
        card.grid(row=0, column=0)

        ttk.Label(card, text="✨ Create Account",
                  font=FONTS["title"],
                  foreground=COLORS["accent"],
                  background=COLORS["bg_card"]).pack(pady=(0, 6))
        ttk.Label(card, text="Join thousands of students worldwide",
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["small"]).pack(pady=(0, 26))

        for lbl, attr, show in [
            ("Display Name", "display_name", ""),
            ("Username",     "username",     ""),
            ("Password",     "password",     "●"),
            ("Confirm Password", "confirm",  "●"),
        ]:
            self._field(card, lbl, attr, show)

        self.error_var = tk.StringVar()
        ttk.Label(card, textvariable=self.error_var,
                  foreground=COLORS["danger"],
                  background=COLORS["bg_card"],
                  font=FONTS["small"]).pack(pady=(4, 0))

        styled_button(card, "Create Account →", self._register,
                      style="accent", width=30).pack(pady=(18, 8))

        back_f = ttk.Frame(card, style="Card.TFrame")
        back_f.pack()
        ttk.Label(back_f, text="Already a member?",
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["body"]).pack(side="left")
        lnk = tk.Label(back_f, text=" Sign in",
                        fg=COLORS["accent_light"],
                        bg=COLORS["bg_card"],
                        font=("Segoe UI", 10, "underline"),
                        cursor="hand2")
        lnk.pack(side="left")
        lnk.bind("<Button-1>", lambda e: self.controller.show_page("LoginPage"))

    def _field(self, parent, label, attr, show=""):
        ttk.Label(parent, text=label,
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["small"]).pack(anchor="w", pady=(10, 2))
        var = tk.StringVar()
        entry = ttk.Entry(parent, textvariable=var, show=show, width=32)
        entry.pack(ipady=6, fill="x")
        setattr(self, f"{attr}_var", var)

    def _register(self):
        dname   = self.display_name_var.get().strip()
        uname   = self.username_var.get().strip().lower()
        pwd     = self.password_var.get().strip()
        confirm = self.confirm_var.get().strip()

        if not all([dname, uname, pwd, confirm]):
            self.error_var.set("All fields are required.")
            return
        if uname in DataStore.users:
            self.error_var.set("Username already taken.")
            return
        if pwd != confirm:
            self.error_var.set("Passwords do not match.")
            return
        if len(pwd) < 6:
            self.error_var.set("Password must be at least 6 characters.")
            return

        DataStore.users[uname] = {"password": pwd, "display_name": dname}
        # Auto-login
        DataStore.current_user = uname
        self.error_var.set("")
        self.controller.show_page("MainApp")

    def on_show(self):
        for attr in ("display_name", "username", "password", "confirm"):
            getattr(self, f"{attr}_var").set("")
        self.error_var.set("")


# ─────────────────────────────────────────────
# MAIN APP  (shell + sidebar + tab pages)
# ─────────────────────────────────────────────
class MainApp(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._build()

    def _build(self):
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)

        # ── Sidebar ──
        self.sidebar = Sidebar(self, self.controller, self._switch_tab)
        self.sidebar.grid(row=0, column=0, sticky="nsew")

        # ── Content area ──
        self.content = ttk.Frame(self, style="Dark.TFrame")
        self.content.grid(row=0, column=1, sticky="nsew")
        self.content.grid_rowconfigure(0, weight=1)
        self.content.grid_columnconfigure(0, weight=1)

        # ── Sub-pages ──
        self.tabs: dict = {}
        for TabClass in (HomePage, ChatPage, EventsPage, CulturePage):
            tab = TabClass(self.content, self.controller)
            self.tabs[TabClass.__name__] = tab
            tab.grid(row=0, column=0, sticky="nsew")

        self._switch_tab("HomePage")

    def _switch_tab(self, tab_name: str):
        self.tabs[tab_name].tkraise()
        if hasattr(self.tabs[tab_name], "on_show"):
            self.tabs[tab_name].on_show()
        self.sidebar.set_active(tab_name)

    def on_show(self):
        self.sidebar.refresh_user()
        self._switch_tab("HomePage")


# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
class Sidebar(ttk.Frame):
    NAV = [
        ("🏠", "Home",    "HomePage"),
        ("💬", "Chat",    "ChatPage"),
        ("🌐", "Events",  "EventsPage"),
        ("🌍", "Cultures","CulturePage"),
    ]

    def __init__(self, parent, controller, switch_fn):
        super().__init__(parent, style="Sidebar.TFrame", width=210)
        self.pack_propagate(False)
        self.controller = controller
        self.switch_fn  = switch_fn
        self.btn_refs   = {}
        self._build()

    def _build(self):
        # Brand
        brand = ttk.Frame(self, style="Sidebar.TFrame", padding=(20, 24, 20, 16))
        brand.pack(fill="x")
        ttk.Label(brand, text="📚 StudyLink",
                  font=FONTS["logo"],
                  style="Sidebar.TLabel").pack(anchor="w")
        ttk.Label(brand, text="Student Platform",
                  style="SidebarMuted.TLabel").pack(anchor="w")

        # Divider
        ttk.Separator(self, orient="horizontal").pack(fill="x", padx=16)

        # Nav buttons
        nav_frame = ttk.Frame(self, style="Sidebar.TFrame", padding=(12, 16, 12, 0))
        nav_frame.pack(fill="x")

        for icon, label, target in self.NAV:
            btn = tk.Button(
                nav_frame,
                text=f"  {icon}  {label}",
                anchor="w",
                font=FONTS["body"],
                bg=COLORS["sidebar"],
                fg=COLORS["text_muted"],
                activebackground=COLORS["bg_hover"],
                activeforeground=COLORS["text_primary"],
                bd=0, relief="flat",
                cursor="hand2",
                padx=12, pady=10,
                command=lambda t=target: self.switch_fn(t),
            )
            btn.pack(fill="x", pady=2)
            self.btn_refs[target] = btn

        # Spacer + user panel at bottom
        bottom = ttk.Frame(self, style="Sidebar.TFrame", padding=16)
        bottom.pack(side="bottom", fill="x")

        ttk.Separator(self, orient="horizontal").pack(side="bottom", fill="x", padx=16)

        # User info
        self.user_frame = ttk.Frame(bottom, style="Sidebar.TFrame")
        self.user_frame.pack(fill="x", pady=(0, 12))
        self._user_widgets()

        # Logout
        tk.Button(
            bottom,
            text="⏻  Logout",
            anchor="w",
            font=FONTS["body"],
            bg=COLORS["sidebar"],
            fg=COLORS["danger"],
            activebackground=COLORS["bg_dark"],
            activeforeground=COLORS["danger"],
            bd=0, relief="flat",
            cursor="hand2",
            padx=12, pady=8,
            command=self._logout,
        ).pack(fill="x")

    def _user_widgets(self):
        for w in self.user_frame.winfo_children():
            w.destroy()
        uname = DataStore.current_user or "guest"
        user  = DataStore.users.get(uname, {})
        dname = user.get("display_name", uname)

        row = ttk.Frame(self.user_frame, style="Sidebar.TFrame")
        row.pack(fill="x")
        av = make_avatar_canvas(row, dname, size=34, bg=COLORS["sidebar"])
        av.pack(side="left", padx=(0, 10))

        info = ttk.Frame(row, style="Sidebar.TFrame")
        info.pack(side="left", fill="x", expand=True)
        ttk.Label(info, text=dname, font=FONTS["body_b"],
                  style="Sidebar.TLabel").pack(anchor="w")
        ttk.Label(info, text=f"@{uname}",
                  style="SidebarMuted.TLabel").pack(anchor="w")

    def refresh_user(self):
        self._user_widgets()

    def set_active(self, target: str):
        for name, btn in self.btn_refs.items():
            if name == target:
                btn.config(bg=COLORS["accent_dim"], fg=COLORS["text_primary"],
                           font=FONTS["body_b"])
            else:
                btn.config(bg=COLORS["sidebar"], fg=COLORS["text_muted"],
                           font=FONTS["body"])

    def _logout(self):
        DataStore.current_user = None
        self.controller.show_page("LoginPage")


# ─────────────────────────────────────────────
# HOME PAGE
# ─────────────────────────────────────────────
class HomePage(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._build()

    def _build(self):
        self.grid_columnconfigure(0, weight=1)
        self.grid_rowconfigure(1, weight=1)

        # Header bar
        header = ttk.Frame(self, style="Card.TFrame", padding=(32, 20))
        header.grid(row=0, column=0, sticky="ew")

        self.greeting_var = tk.StringVar(value="Welcome back! 👋")
        ttk.Label(header, textvariable=self.greeting_var,
                  font=FONTS["title"],
                  background=COLORS["bg_card"],
                  foreground=COLORS["text_primary"]).pack(anchor="w")
        ttk.Label(header, text="Here's what's happening in your community today.",
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["body"]).pack(anchor="w", pady=(4, 0))

        # Scrollable content
        canvas = tk.Canvas(self, bg=COLORS["bg_dark"], highlightthickness=0)
        scroll = ttk.Scrollbar(self, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=scroll.set)
        canvas.grid(row=1, column=0, sticky="nsew")
        scroll.grid(row=1, column=1, sticky="ns")

        inner = ttk.Frame(canvas, style="Dark.TFrame", padding=(32, 24))
        canvas.create_window((0, 0), window=inner, anchor="nw", tags="inner")

        def _resize(e):
            canvas.configure(scrollregion=canvas.bbox("all"))
            canvas.itemconfig("inner", width=e.width)
        canvas.bind("<Configure>", _resize)
        inner.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))

        inner.grid_columnconfigure(0, weight=1)
        inner.grid_columnconfigure(1, weight=1)

        # ── Stats row ──
        stats = [
            ("👥", f"{len(DataStore.users)}", "Students online"),
            ("💬", f"{len(DataStore.messages)}", "Messages sent"),
            ("🌐", f"{len(DataStore.EVENTS)}", "Upcoming events"),
            ("🌍", f"{len(DataStore.CULTURES)}", "Cultures explored"),
        ]
        stats_frame = ttk.Frame(inner, style="Dark.TFrame")
        stats_frame.grid(row=0, column=0, columnspan=2, sticky="ew", pady=(0, 28))
        stats_frame.grid_columnconfigure(tuple(range(4)), weight=1)

        for i, (icon, val, lbl) in enumerate(stats):
            card = ttk.Frame(stats_frame, style="Card.TFrame", padding=(20, 18))
            card.grid(row=0, column=i, sticky="ew", padx=(0, 12 if i < 3 else 0))
            ttk.Label(card, text=icon, font=("Segoe UI", 20),
                      background=COLORS["bg_card"]).pack(anchor="w")
            ttk.Label(card, text=val, font=("Segoe UI", 22, "bold"),
                      foreground=COLORS["accent"],
                      background=COLORS["bg_card"]).pack(anchor="w")
            ttk.Label(card, text=lbl, style="Muted.TLabel",
                      background=COLORS["bg_card"]).pack(anchor="w")

        # ── Quick actions ──
        ttk.Label(inner, text="Quick Actions",
                  font=FONTS["heading"],
                  foreground=COLORS["text_primary"]).grid(
            row=1, column=0, columnspan=2, sticky="w", pady=(0, 12))

        actions_frame = ttk.Frame(inner, style="Dark.TFrame")
        actions_frame.grid(row=2, column=0, columnspan=2, sticky="ew", pady=(0, 28))
        actions_frame.grid_columnconfigure((0, 1, 2), weight=1)

        self._action_card(actions_frame, 0, "💬", "Open Chat",
                          "Message your classmates", "ChatPage")
        self._action_card(actions_frame, 1, "🌐", "Browse Events",
                          "Find global opportunities", "EventsPage")
        self._action_card(actions_frame, 2, "🌍", "Learn Cultures",
                          "Discover the world", "CulturePage")

        # ── Recent messages preview ──
        ttk.Label(inner, text="Recent Chat",
                  font=FONTS["heading"],
                  foreground=COLORS["text_primary"]).grid(
            row=3, column=0, columnspan=2, sticky="w", pady=(0, 12))

        self.msg_preview = ttk.Frame(inner, style="Card.TFrame", padding=16)
        self.msg_preview.grid(row=4, column=0, columnspan=2, sticky="ew")

    def _action_card(self, parent, col, icon, title, sub, target):
        card = ttk.Frame(parent, style="Card.TFrame", padding=(20, 18))
        card.grid(row=0, column=col, sticky="ew",
                  padx=(0, 12 if col < 2 else 0))
        ttk.Label(card, text=icon, font=("Segoe UI", 22),
                  background=COLORS["bg_card"]).pack(anchor="w")
        ttk.Label(card, text=title, font=FONTS["body_b"],
                  foreground=COLORS["text_primary"],
                  background=COLORS["bg_card"]).pack(anchor="w", pady=(6, 2))
        ttk.Label(card, text=sub, style="Muted.TLabel",
                  background=COLORS["bg_card"]).pack(anchor="w")
        styled_button(card, "Open →",
                      lambda t=target: self._go(t),
                      style="ghost").pack(anchor="w", pady=(10, 0))

    def _go(self, target):
        # Navigate via MainApp
        main = self.controller.pages["MainApp"]
        main._switch_tab(target)

    def on_show(self):
        uname = DataStore.current_user or "Student"
        user  = DataStore.users.get(uname, {})
        dname = user.get("display_name", uname)
        self.greeting_var.set(f"Welcome back, {dname}! 👋")
        self._refresh_preview()

    def _refresh_preview(self):
        for w in self.msg_preview.winfo_children():
            w.destroy()
        msgs = DataStore.messages[-4:]
        if not msgs:
            ttk.Label(self.msg_preview, text="No messages yet. Say hello!",
                      foreground=COLORS["text_muted"],
                      background=COLORS["bg_card"]).pack(pady=10)
            return
        for msg in reversed(msgs):
            row = ttk.Frame(self.msg_preview, style="Card.TFrame")
            row.pack(fill="x", pady=3)
            user = DataStore.users.get(msg["sender"], {})
            dname = user.get("display_name", msg["sender"])
            av = make_avatar_canvas(row, dname, size=28, bg=COLORS["bg_card"])
            av.pack(side="left", padx=(0, 8))
            info = ttk.Frame(row, style="Card.TFrame")
            info.pack(side="left", fill="x", expand=True)
            ttk.Label(info, text=dname, font=FONTS["body_b"],
                      foreground=COLORS["text_primary"],
                      background=COLORS["bg_card"]).pack(anchor="w")
            preview = msg["text"][:60] + ("…" if len(msg["text"]) > 60 else "")
            ttk.Label(info, text=preview, foreground=COLORS["text_muted"],
                      background=COLORS["bg_card"],
                      font=FONTS["small"]).pack(anchor="w")
            ttk.Label(row, text=msg["timestamp"],
                      foreground=COLORS["text_dim"],
                      background=COLORS["bg_card"],
                      font=FONTS["small"]).pack(side="right", padx=8)


# ─────────────────────────────────────────────
# CHAT PAGE  (WhatsApp-style)
# ─────────────────────────────────────────────
class ChatPage(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._typing_job = None
        self._build()

    def _build(self):
        self.grid_rowconfigure(1, weight=1)
        self.grid_columnconfigure(0, weight=1)

        # ── Chat header ──
        header = ttk.Frame(self, style="Card.TFrame", padding=(20, 14))
        header.grid(row=0, column=0, sticky="ew")
        ttk.Label(header, text="💬  Global Chat",
                  font=FONTS["heading"],
                  background=COLORS["bg_card"],
                  foreground=COLORS["text_primary"]).pack(side="left")
        self.online_lbl = ttk.Label(header,
                                    text=f"🟢 {len(DataStore.users)} online",
                                    foreground=COLORS["success"],
                                    background=COLORS["bg_card"],
                                    font=FONTS["small"])
        self.online_lbl.pack(side="right")

        # ── Messages canvas ──
        self.msg_canvas = tk.Canvas(self, bg=COLORS["bg_dark"],
                                    highlightthickness=0)
        vscroll = ttk.Scrollbar(self, orient="vertical",
                                 command=self.msg_canvas.yview)
        self.msg_canvas.configure(yscrollcommand=vscroll.set)
        self.msg_canvas.grid(row=1, column=0, sticky="nsew")
        vscroll.grid(row=1, column=1, sticky="ns")

        self.msg_inner = ttk.Frame(self.msg_canvas, style="Dark.TFrame",
                                   padding=(16, 12))
        self.msg_canvas.create_window((0, 0), window=self.msg_inner,
                                      anchor="nw", tags="inner")

        def _on_resize(e):
            self.msg_canvas.itemconfig("inner", width=e.width)
            self.msg_canvas.configure(scrollregion=self.msg_canvas.bbox("all"))
        self.msg_canvas.bind("<Configure>", _on_resize)
        self.msg_inner.bind("<Configure>",
                            lambda e: self.msg_canvas.configure(
                                scrollregion=self.msg_canvas.bbox("all")))

        # Scroll with mouse-wheel
        self.msg_canvas.bind_all("<MouseWheel>", self._on_mousewheel)

        # ── Typing indicator ──
        self.typing_var = tk.StringVar(value="")
        self.typing_lbl = ttk.Label(self, textvariable=self.typing_var,
                                    foreground=COLORS["text_muted"],
                                    font=("Segoe UI", 9, "italic"))
        self.typing_lbl.grid(row=2, column=0, sticky="w", padx=20, pady=(4, 0))

        # ── Input area ──
        input_frame = ttk.Frame(self, style="Card.TFrame", padding=(16, 12))
        input_frame.grid(row=3, column=0, sticky="ew", columnspan=2)
        input_frame.grid_columnconfigure(0, weight=1)

        self.msg_var = tk.StringVar()
        entry = ttk.Entry(input_frame, textvariable=self.msg_var,
                          font=FONTS["body"])
        entry.grid(row=0, column=0, sticky="ew", ipady=8, padx=(0, 12))
        entry.bind("<Return>", lambda e: self._send())
        entry.bind("<KeyRelease>", self._on_key)
        self.entry = entry

        styled_button(input_frame, "Send ➤", self._send,
                      style="success").grid(row=0, column=1)

    def _on_mousewheel(self, event):
        self.msg_canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")

    def _render_messages(self):
        for w in self.msg_inner.winfo_children():
            w.destroy()
        cur = DataStore.current_user
        for msg in DataStore.messages:
            self._add_bubble(msg, is_self=(msg["sender"] == cur))
        self._scroll_to_bottom()

    def _add_bubble(self, msg: dict, is_self: bool):
        user  = DataStore.users.get(msg["sender"], {})
        dname = user.get("display_name", msg["sender"])

        outer = ttk.Frame(self.msg_inner, style="Dark.TFrame")
        outer.pack(fill="x", pady=5)

        if is_self:
            side_frame = ttk.Frame(outer, style="Dark.TFrame")
            side_frame.pack(anchor="e")
            self._bubble_content(side_frame, msg, dname, is_self=True)
        else:
            side_frame = ttk.Frame(outer, style="Dark.TFrame")
            side_frame.pack(anchor="w")
            # Avatar left
            av = make_avatar_canvas(side_frame, dname, size=34,
                                    bg=COLORS["bg_dark"])
            av.pack(side="left", anchor="n", padx=(0, 8))
            self._bubble_content(side_frame, msg, dname, is_self=False)

    def _bubble_content(self, parent, msg, dname, is_self):
        bubble_bg   = COLORS["bubble_self"]   if is_self else COLORS["bubble_other"]
        bubble_fg   = COLORS["bubble_self_text"] if is_self else COLORS["bubble_other_text"]
        anchor_side = "e" if is_self else "w"

        wrapper = tk.Frame(parent, bg=COLORS["bg_dark"])
        wrapper.pack(side="left" if not is_self else "right")

        # Sender name (only for others)
        if not is_self:
            tk.Label(wrapper, text=dname, bg=COLORS["bg_dark"],
                     fg=COLORS["accent_light"],
                     font=("Segoe UI", 8, "bold")).pack(anchor="w",
                                                         padx=(12, 12))

        bubble = tk.Frame(wrapper, bg=bubble_bg,
                          padx=14, pady=10)
        bubble.pack(anchor=anchor_side)

        # Message text (word-wrap via label)
        tk.Label(bubble, text=msg["text"],
                 bg=bubble_bg, fg=bubble_fg,
                 font=FONTS["bubble"],
                 wraplength=340, justify="left").pack(anchor="w")

        # Timestamp + delivered tick
        meta = tk.Frame(bubble, bg=bubble_bg)
        meta.pack(anchor="e", pady=(4, 0))
        tk.Label(meta, text=msg["timestamp"],
                 bg=bubble_bg, fg=bubble_fg if is_self else COLORS["text_muted"],
                 font=("Segoe UI", 7)).pack(side="left")
        if is_self and msg.get("delivered"):
            tk.Label(meta, text=" ✓",
                     bg=bubble_bg,
                     fg=bubble_fg,
                     font=("Segoe UI", 7, "bold")).pack(side="left")

    def _send(self):
        text = self.msg_var.get().strip()
        if not text:
            return
        DataStore.add_message(DataStore.current_user, text)
        self.msg_var.set("")
        self.typing_var.set("")
        self._render_messages()
        # Simulate reply after short delay
        self.after(1500, self._simulate_reply)

    def _simulate_reply(self):
        others = [u for u in DataStore.users if u != DataStore.current_user]
        if not others:
            return
        other = random.choice(others)
        dname = DataStore.users[other]["display_name"]
        replies = [
            "That's a great point! 🤔",
            "I totally agree with that!",
            "Can you tell me more?",
            "Nice one! 🎉",
            "Let's discuss this further in the events page!",
            "I was just thinking about that 😄",
        ]
        # Show typing first
        self.typing_var.set(f"{dname} is typing...")
        self.after(1200, lambda: self._finish_reply(other))

    def _finish_reply(self, sender):
        replies = [
            "That's a great point! 🤔",
            "I totally agree with that!",
            "Can you tell me more?",
            "Nice one! 🎉",
            "Let's discuss this further in the events page!",
            "I was just thinking about that 😄",
        ]
        DataStore.add_message(sender, random.choice(replies))
        self.typing_var.set("")
        self._render_messages()

    def _on_key(self, event):
        """Show typing indicator while user types."""
        user  = DataStore.users.get(DataStore.current_user, {})
        dname = user.get("display_name", DataStore.current_user or "You")
        if self.msg_var.get():
            self.typing_var.set(f"{dname} is typing...")
        else:
            self.typing_var.set("")

    def _scroll_to_bottom(self):
        self.msg_canvas.update_idletasks()
        self.msg_canvas.yview_moveto(1.0)

    def on_show(self):
        self.online_lbl.config(text=f"🟢 {len(DataStore.users)} online")
        self._render_messages()


# ─────────────────────────────────────────────
# EVENTS PAGE
# ─────────────────────────────────────────────
class EventsPage(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._build()

    def _build(self):
        self.grid_rowconfigure(1, weight=1)
        self.grid_columnconfigure(0, weight=1)

        header = ttk.Frame(self, style="Card.TFrame", padding=(28, 20))
        header.grid(row=0, column=0, sticky="ew")
        ttk.Label(header, text="🌐  Worldwide Events",
                  font=FONTS["title"],
                  background=COLORS["bg_card"],
                  foreground=COLORS["text_primary"]).pack(anchor="w")
        ttk.Label(header,
                  text="Discover global opportunities — conferences, hackathons & more.",
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["body"]).pack(anchor="w", pady=(4, 0))

        # Scrollable grid
        canvas = tk.Canvas(self, bg=COLORS["bg_dark"], highlightthickness=0)
        scroll = ttk.Scrollbar(self, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=scroll.set)
        canvas.grid(row=1, column=0, sticky="nsew")
        scroll.grid(row=1, column=1, sticky="ns")

        inner = ttk.Frame(canvas, style="Dark.TFrame", padding=(28, 20))
        canvas.create_window((0, 0), window=inner, anchor="nw", tags="inner")

        def _resize(e):
            canvas.configure(scrollregion=canvas.bbox("all"))
            canvas.itemconfig("inner", width=e.width)
        canvas.bind("<Configure>", _resize)
        inner.bind("<Configure>",
                   lambda e: canvas.configure(scrollregion=canvas.bbox("all")))

        inner.grid_columnconfigure(0, weight=1)
        inner.grid_columnconfigure(1, weight=1)

        for i, event in enumerate(DataStore.EVENTS):
            row, col = divmod(i, 2)
            self._event_card(inner, event, row, col)

    def _event_card(self, parent, event, row, col):
        card = ttk.Frame(parent, style="Card.TFrame", padding=(22, 20))
        card.grid(row=row, column=col, sticky="ew",
                  padx=(0, 12 if col == 0 else 0), pady=(0, 14))
        card.grid_columnconfigure(1, weight=1)

        # Icon
        tk.Label(card, text=event["emoji"], font=("Segoe UI", 28),
                 bg=COLORS["bg_card"]).grid(row=0, column=0, rowspan=2,
                                             sticky="n", padx=(0, 14))

        # Title
        ttk.Label(card, text=event["title"], font=FONTS["body_b"],
                  foreground=COLORS["text_primary"],
                  background=COLORS["bg_card"]).grid(row=0, column=1,
                                                      sticky="w")

        # Date & Location badge
        meta = ttk.Frame(card, style="Card.TFrame")
        meta.grid(row=1, column=1, sticky="w", pady=(4, 8))
        for txt, color in [(f"📅 {event['date']}", COLORS["accent_light"]),
                           (f"  📍 {event['loc']}", COLORS["text_muted"])]:
            ttk.Label(meta, text=txt, foreground=color,
                      background=COLORS["bg_card"],
                      font=FONTS["small"]).pack(side="left")

        # Description
        ttk.Label(card, text=event["desc"],
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  wraplength=320, justify="left",
                  font=FONTS["body"]).grid(row=2, column=0, columnspan=2,
                                           sticky="w", pady=(0, 12))

        styled_button(card, "Learn More →", lambda: None,
                      style="ghost").grid(row=3, column=0, sticky="w")


# ─────────────────────────────────────────────
# CULTURE PAGE
# ─────────────────────────────────────────────
class CulturePage(BasePage):
    def __init__(self, parent, controller):
        super().__init__(parent, controller)
        self._build()

    def _build(self):
        self.grid_rowconfigure(1, weight=1)
        self.grid_columnconfigure(0, weight=1)

        header = ttk.Frame(self, style="Card.TFrame", padding=(28, 20))
        header.grid(row=0, column=0, sticky="ew")
        ttk.Label(header, text="🌍  Learn Cultures",
                  font=FONTS["title"],
                  background=COLORS["bg_card"],
                  foreground=COLORS["text_primary"]).pack(anchor="w")
        ttk.Label(header,
                  text="Expand your worldview — one country at a time.",
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  font=FONTS["body"]).pack(anchor="w", pady=(4, 0))

        canvas = tk.Canvas(self, bg=COLORS["bg_dark"], highlightthickness=0)
        scroll = ttk.Scrollbar(self, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=scroll.set)
        canvas.grid(row=1, column=0, sticky="nsew")
        scroll.grid(row=1, column=1, sticky="ns")

        inner = ttk.Frame(canvas, style="Dark.TFrame", padding=(28, 20))
        canvas.create_window((0, 0), window=inner, anchor="nw", tags="inner")

        def _resize(e):
            canvas.configure(scrollregion=canvas.bbox("all"))
            canvas.itemconfig("inner", width=e.width)
        canvas.bind("<Configure>", _resize)
        inner.bind("<Configure>",
                   lambda e: canvas.configure(scrollregion=canvas.bbox("all")))

        inner.grid_columnconfigure(0, weight=1)
        inner.grid_columnconfigure(1, weight=1)

        for i, culture in enumerate(DataStore.CULTURES):
            row, col = divmod(i, 2)
            self._culture_card(inner, culture, row, col)

    def _culture_card(self, parent, c, row, col):
        card = ttk.Frame(parent, style="Card.TFrame", padding=(22, 18))
        card.grid(row=row, column=col, sticky="ew",
                  padx=(0, 12 if col == 0 else 0), pady=(0, 14))

        # Flag + Country
        top = ttk.Frame(card, style="Card.TFrame")
        top.pack(fill="x")
        tk.Label(top, text=c["flag"], font=("Segoe UI", 26),
                 bg=COLORS["bg_card"]).pack(side="left", padx=(0, 10))

        name_block = ttk.Frame(top, style="Card.TFrame")
        name_block.pack(side="left")
        ttk.Label(name_block, text=c["country"], font=FONTS["heading"],
                  foreground=COLORS["text_primary"],
                  background=COLORS["bg_card"]).pack(anchor="w")

        # Greeting badge
        badge = tk.Label(name_block, text=f'  {c["greeting"]}  ',
                         bg=c["color"], fg="white",
                         font=("Segoe UI", 9, "bold"),
                         padx=6, pady=2)
        badge.pack(anchor="w", pady=(3, 0))

        # Fact
        ttk.Separator(card, orient="horizontal").pack(fill="x", pady=10)
        ttk.Label(card, text="📌  " + c["fact"],
                  foreground=COLORS["text_muted"],
                  background=COLORS["bg_card"],
                  wraplength=310, justify="left",
                  font=FONTS["body"]).pack(anchor="w")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    app = App()
    app.mainloop()